# Phase 6 — check the six runs before anything else

Reads files. Trains nothing. **Turn the accelerator OFF** — there is no reason to spend
GPU on this.

It answers the two questions that must be settled before the unblinding:

1. **Are all six runs finished, and did any collapse?** A run directory proves only that
   a run *started*. `metrics.json` is written after the last epoch, so a run killed
   part-way leaves `checkpoint.pt` and nothing else.
2. **Which EyePACS split did the models train on?** The variants were built with
   `--eyepacs-split regroup`. This measures how many official-test images ended up in
   training, and whether a protocol-matched leaderboard model (option C) is possible.

## Inputs

`verify-dr-phase6` and `verify-dr-manifests`. Nothing else — no cache, no repo clone.

| Setting | Value |
|---|---|
| Accelerator | **None** |
| Internet | Off is fine |

Copy **both** sections' output back.

In [ ]:
import json
from pathlib import Path

import pandas as pd

INPUT = Path('/kaggle/input')
RUNS = [f'H1_{v}_s{s}' for v in ('eyepacs_full', 'eyepacs_ddr_full') for s in (42, 43, 44)]


def first(pattern):
    hits = sorted(INPUT.rglob(pattern))
    return hits[0] if hits else None

## 1 · The six runs

In [ ]:
# ---- 1. Are all six runs finished -- and did any collapse? --------------------
# A directory existing proves only that a run STARTED. metrics.json is written
# once, after the last epoch; a run killed mid-way leaves checkpoint.pt alone.
done = {}
for p in sorted(INPUT.rglob('H1_*/metrics.json')):
    done.setdefault(p.parent.name, p)
started = {p.parent.name for p in INPUT.rglob('H1_*/checkpoint.pt')}

rows = []
for name in RUNS:
    if name not in done:
        rows.append({'run': name,
                     'status': 'PARTIAL - resume it' if name in started else 'MISSING'})
        continue
    m = json.loads(done[name].read_text())
    b = m.get('best_val') or {}
    rows.append({
        'run': name, 'status': 'done',
        'epochs': m.get('epochs_run'), 'best ep': m.get('best_epoch'),
        'QWK': round(b.get('qwk', float('nan')), 4),
        'macro-F1': round(b.get('macro_f1', float('nan')), 4),
        'g1-F1': round(b.get('per_class_f1', {}).get('1', float('nan')), 4),
        'distinct': b.get('distinct_predictions'),
    })

print('=' * 78)
print('1. PHASE 6a RUNS')
print('=' * 78)
table = pd.DataFrame(rows)
for col in ('epochs', 'best ep', 'distinct'):      # NaN from unfinished rows forces floats
    if col in table:
        table[col] = table[col].astype('Int64')
print(table.to_string(index=False))
n_done = sum(r['status'] == 'done' for r in rows)
collapsed = [r['run'] for r in rows if r.get('distinct') == 1]
print()
if n_done < len(RUNS):
    print(f'{len(RUNS) - n_done} run(s) not finished. Re-open 06_final_training with')
    print('verify-dr-phase6 attached; section 6 carries them forward and --resume')
    print('continues each from its last epoch.')
    if any(r['status'] == 'MISSING' for r in rows):
        print()
        print('MISSING but visible in 06_final_training\'s Output tab? Then the attached')
        print('verify-dr-phase6 is an OLDER version. Update the dataset to the latest')
        print('notebook version, re-attach, and re-run this.')
if collapsed:
    print(f'COLLAPSED (predicts one grade for every image): {collapsed}')
    print('These decide nothing, whatever their QWK. They must be retrained.')
if n_done == len(RUNS) and not collapsed:
    print('All six complete and none collapsed. Phase 6a training is DONE.')

## 2 · The EyePACS split

In [ ]:
# ---- 2. Which EyePACS split did the six models actually train on? -------------
print()
print('=' * 78)
print('2. THE EYEPACS SPLIT')
print('=' * 78)
plan = first('dataset_plan.json')
if plan is not None:
    policy = json.loads(plan.read_text()).get('eyepacs', {}).get('split_policy', {})
    print('dataset_plan.json records:', policy)
else:
    print('dataset_plan.json not found -- attach verify-dr-manifests')

full = first('eyepacs_full.csv')
if full is None:
    print('eyepacs_full.csv not found -- attach verify-dr-manifests')
else:
    f = pd.read_csv(full, usecols=lambda c: c in ('split', 'source_split'))
    src = (f['source_split'].fillna('').astype(str) if 'source_split' in f.columns
           else pd.Series('', index=f.index))
    if not src.ne('').any():
        print()
        print('No source_split recorded: the mirror did not say which images were the')
        print('official test set. They cannot be identified, so a protocol-matched')
        print('leaderboard number is impossible from this cache -> OPTION A.')
    else:
        print()
        print('rows = the mirror\'s own split; columns = the split we trained with')
        print(pd.crosstab(src.replace('', '(none)'), f['split'], margins=True))
        # The competition's own partition: 35,126 train / 53,576 test, and no val.
        # A mirror that re-split the data still says "train" and "test" -- the
        # labels alone prove nothing, so check the counts before trusting them.
        # (The cache is 693 images short of 88,702, hence the 2% tolerance.)
        OFFICIAL = {'train': 35126, 'test': 53576}
        counts = src[src != ''].value_counts()
        is_official = (set(counts.index) <= set(OFFICIAL) and all(
            abs(int(counts.get(k, 0)) - n) <= 0.02 * n for k, n in OFFICIAL.items()))
        print()
        print("the mirror's split:  " + '  '.join(f'{k} {int(v):,}' for k, v in counts.items()))
        print("the competition's:   train 35,126  test 53,576  (no val)")

        # Our regroup ignored every external partition, so ANY independently chosen
        # image set -- the official test set included -- has about the same share
        # of its images in our train/val/calibration as the whole cache does.
        dev = float(f['split'].isin(['train', 'val', 'calibration']).mean())

        if not is_official:
            print()
            print("NOT THE OFFICIAL SPLIT. The mirror re-split the data itself, so its")
            print("'test' is not the competition's test set. OPTION C is IMPOSSIBLE from")
            print("this manifest: the official partition cannot be recovered from it.")
            print("It would need the competition's own trainLabels.csv (35,126 IDs).")
            print()
            print(f"Expected overlap anyway: {dev:.0%} of all EyePACS images are in our")
            print("train/val/calibration, and the regroup ignored the official partition,")
            print(f"so about {dev:.0%} of the official test set is there too. Scoring the six")
            print("models on it would largely score them on their own training data.")
        else:
            official_test = f[src == 'test']
            seen = int(official_test['split'].isin(['train', 'val', 'calibration']).sum())
            print()
            print(f"The mirror's split matches the competition's. Of its {len(official_test):,}")
            print(f"test images, {seen:,} ({seen / len(official_test):.0%}) are in our "
                  f"train/val/calibration splits.")
            n_train = int((src == 'train').sum())
            fr = json.loads(plan.read_text()).get('eyepacs', {}) if plan else {}
            keep = 1 - fr.get('val_frac', 0.10) - fr.get('calibration_frac', 0.05)
            h = n_train * keep * 10 / 46.0 / 3600
            print()
            print(f'OPTION C is possible: {n_train:,} official-train images, '
                  f'~{n_train * keep:,.0f}')
            print(f'after carving val/calibration -> ~{h:.1f} GPU-h per seed '
                  f'at the frozen recipe.')


---
**Nothing locked is read here.** `eyepacs_full.csv` is a manifest of paths and splits;
the check reads only its `split` and `source_split` columns — no grades, no images, and
nothing from APTOS or Messidor-2.